In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/06 10:18:16 WARN Utils: Your hostname, DESKTOP-SAJRLCQ, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/06 10:18:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/06 10:18:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# !wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

In [4]:
# !gzip -d fhvhv_tripdata_2021-01.csv.gz

In [5]:
# !wc -l fhvhv_tripdata_2021-01.csv

In [6]:
df = spark.read \
    .option("header", "true") \
    .csv('fhvhv_tripdata_2021-01.csv')

In [7]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   NULL|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   NULL|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   NULL|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [8]:
df.head(5)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime='2021-01-01 00:33:44', dropoff_datetime='2021-01-01 00:49:07', PULocationID='230', DOLocationID='166', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime='2021-01-01 00:55:19', dropoff_datetime='2021-01-01 01:18:21', PULocationID='152', DOLocationID='167', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:23:56', dropoff_datetime='2021-01-01 00:38:05', PULocationID='233', DOLocationID='142', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:42:51', dropoff_datetime='2021-01-01 00:45:50', PULocationID='142', DOLocationID='143', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:48:14', dropoff_datetime='2021-01-01 01:08:42', PULocationID='143', DOLocationID='78', SR_Flag=None)]

In [9]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [15]:
!head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv

In [16]:
!wc -l head.csv

1001 head.csv


In [18]:
import pandas as pd

In [20]:
df_pandas = pd.read_csv('head.csv') 

In [22]:
df_pandas.dtypes

hvfhs_license_num           str
dispatching_base_num        str
pickup_datetime             str
dropoff_datetime            str
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [26]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

In [23]:
from pyspark.sql import types

In [28]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [29]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv')

In [30]:
df.head(10)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 33, 44), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 49, 7), PULocationID=230, DOLocationID=166, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 55, 19), dropoff_datetime=datetime.datetime(2021, 1, 1, 1, 18, 21), PULocationID=152, DOLocationID=167, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 23, 56), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 38, 5), PULocationID=233, DOLocationID=142, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 42, 51), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 45, 50), PULocationID=142, DOLocationID=143, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_dat

In [31]:
df.repartition(24) 

DataFrame[hvfhs_license_num: string, dispatching_base_num: string, pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int, SR_Flag: string]

In [10]:
df.write.parquet('fhvhv/2021/01/')

AnalysisException: [PATH_ALREADY_EXISTS] Path file:/home/ethan/data-engineering-zoomcamp/spark/fhvhv/2021/01 already exists. Set mode as "overwrite" to overwrite the existing path. SQLSTATE: 42K04

In [11]:
df= spark.read.parquet('fhvhv/2021/01/')

In [13]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [16]:
df.select('pickup_datetime','dropoff_datetime','PULocationID','DOLocationID') \
    .filter(df.hvfhs_license_num == 'HV0003') \
    .show()

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-03 00:29:29|2021-01-03 00:46:37|         197|          82|
|2021-01-03 00:50:22|2021-01-03 00:52:52|          56|          56|
|2021-01-03 00:05:27|2021-01-03 00:15:20|         243|         244|
|2021-01-03 00:24:41|2021-01-03 00:30:47|         243|         127|
|2021-01-03 00:48:14|2021-01-03 01:00:10|         235|          18|
|2021-01-03 00:19:02|2021-01-03 00:28:57|         151|         116|
|2021-01-03 00:09:51|2021-01-03 00:19:16|         121|          28|
|2021-01-03 00:30:20|2021-01-03 00:45:24|          28|         160|
|2021-01-03 00:47:31|2021-01-03 00:56:37|         160|         157|
|2021-01-03 00:26:17|2021-01-03 00:50:36|         236|         265|
|2021-01-03 00:10:04|2021-01-03 00:14:38|           3|          32|
|2021-01-03 00:41:48|2021-01-03 00:48:48|       

In [17]:
from pyspark.sql import functions as F

In [19]:
df.withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .select('pickup_date','dropoff_date','PULocationID','DOLocationID')\
    .show()


+-----------+------------+------------+------------+
|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-----------+------------+------------+------------+
| 2021-01-03|  2021-01-03|         197|          82|
| 2021-01-03|  2021-01-03|          56|          56|
| 2021-01-03|  2021-01-03|          76|          51|
| 2021-01-03|  2021-01-03|         208|         208|
| 2021-01-03|  2021-01-03|         183|         208|
| 2021-01-03|  2021-01-03|         243|         244|
| 2021-01-03|  2021-01-03|         243|         127|
| 2021-01-03|  2021-01-03|         235|          18|
| 2021-01-03|  2021-01-03|          68|          49|
| 2021-01-03|  2021-01-03|         151|         116|
| 2021-01-03|  2021-01-03|         121|          28|
| 2021-01-03|  2021-01-03|          28|         160|
| 2021-01-03|  2021-01-03|         160|         157|
| 2021-01-03|  2021-01-03|         236|         265|
| 2021-01-03|  2021-01-03|          32|         169|
| 2021-01-03|  2021-01-03|           3|       

In [20]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 ==0:
        return f's/{num:03x}'
    else:
        return f'e/{num:03x}'

In [21]:
crazy_stuff('B02884')

's/b44'

In [24]:
crazy_stuff_udf = F.udf(crazy_stuff,returnType=types.StringType())

In [26]:
df.withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id',crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id','pickup_date','dropoff_date','PULocationID','DOLocationID')\
    .show()


+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/acc| 2021-01-03|  2021-01-03|         197|          82|
|  e/acc| 2021-01-03|  2021-01-03|          56|          56|
|  e/9ce| 2021-01-03|  2021-01-03|          76|          51|
|  e/9ce| 2021-01-03|  2021-01-03|         208|         208|
|  e/9ce| 2021-01-03|  2021-01-03|         183|         208|
|  e/b14| 2021-01-03|  2021-01-03|         243|         244|
|  e/b14| 2021-01-03|  2021-01-03|         243|         127|
|  e/b14| 2021-01-03|  2021-01-03|         235|          18|
|  e/9ce| 2021-01-03|  2021-01-03|          68|          49|
|  e/b35| 2021-01-03|  2021-01-03|         151|         116|
|  e/b32| 2021-01-03|  2021-01-03|         121|          28|
|  e/b32| 2021-01-03|  2021-01-03|          28|         160|
|  e/b32| 2021-01-03|  2021-01-03|         160|         157|
|  e/b35| 2021-01-03|  2